# E-commerce Promo Code & Retention Analysis

本 Notebook 用于作品集展示：读取真实 Kaggle CSV，完成数据验证、月度指标、促销码对比、first observed cohort 留存、RFM 分层、促销敏感度和 180 天未来不活跃风险建模。

当前主数据文件为 `data/raw/EcommData_CSV.csv`，清洗后包含 102,771 条订单、3,900 个客户，日期范围为 2022-01-01 至 2024-12-31。

重要边界：促销码分析是相关性分析，不声称优惠券导致留存或流失变化。`Previous Purchases` 显示客户在数据窗口前可能已有购买历史，因此 cohort 代表 first observed purchase cohort，不代表生命周期真实首购。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from promo_retention.data import load_raw_csv, clean_transactions
from promo_retention.validation import validate_dataset, summarize_cleaning
from promo_retention.metrics import monthly_kpis, promo_comparison, cohort_retention, cohort_matrix, rfm_segments, segment_summary, promo_sensitivity_summary, segment_promo_matrix
from promo_retention.plotting import save_monthly_trend, save_cohort_heatmap, save_promo_comparison, save_segment_matrix
from promo_retention.modeling import build_inactivity_snapshot, train_inactivity_risk_model

DATA_PATH = PROJECT_ROOT / 'data/raw/EcommData_CSV.csv'
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)

raw = load_raw_csv(DATA_PATH)
df = clean_transactions(raw)
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,promo_code_used,previous_purchases,payment_method,purchase_date,weekdaynum,weekday,weekend,churn,order_id,purchase_month,purchase_quarter,promo_segment,first_observed_purchase_date,first_observed_purchase_month,first_purchase_date,first_purchase_month,cohort_index
0,1,55,Male,Belt,Accessories,46.9,Kentucky,L,Gray,Winter,5.0,1,Standard,0,14,Credit Card,2022-01-07,5,Friday,0,1,1,2022-01-01,2022Q1,No Promo,2022-01-07,2022-01-01,2022-01-07,2022-01-01,0
1,1,55,Male,Sweater,Clothing,48.1,Kentucky,L,Green,Spring,1.0,1,Free Shipping,0,14,Credit Card,2022-03-19,6,Saturday,1,1,2,2022-03-01,2022Q1,No Promo,2022-01-07,2022-01-01,2022-01-07,2022-01-01,2
2,1,55,Male,Sweater,Clothing,62.1,Kentucky,L,Green,Spring,5.0,1,Next Day Air,0,14,Debit Card,2022-05-03,2,Tuesday,0,1,3,2022-05-01,2022Q2,No Promo,2022-01-07,2022-01-01,2022-01-07,2022-01-01,4
3,1,55,Male,Belt,Accessories,25.0,Kentucky,L,Turquoise,Summer,5.0,1,Free Shipping,0,14,Credit Card,2022-05-07,6,Saturday,1,1,4,2022-05-01,2022Q2,No Promo,2022-01-07,2022-01-01,2022-01-07,2022-01-01,4
4,1,55,Male,Sweater,Clothing,89.8,Kentucky,L,Blue,Summer,5.0,1,2-Day Shipping,0,14,Google Pay,2022-06-10,5,Friday,0,1,5,2022-06-01,2022Q2,No Promo,2022-01-07,2022-01-01,2022-01-07,2022-01-01,5


## 1. Data Validation

In [2]:
validate_dataset(df)

,check,status,value,missing_rate
0,row_count,PASS,102771,0.0
1,customer_count,PASS,3900,0.0
2,required_column:customer_id,PASS,,0.0
3,required_column:purchase_date,PASS,,0.0
4,required_column:purchase_amount,PASS,,0.0
5,required_column:promo_code_used,PASS,,0.0
6,required_column:previous_purchases,PASS,,0.0
7,required_column:churn,PASS,,0.0
8,date_range,PASS,2022-01-01 to 2024-12-31,0.0
9,positive_purchase_amount,PASS,0,0.0


In [3]:
summarize_cleaning(raw, df)

,stage,rows,customers,sales,promo_usage_rate
0,raw,102771,3900,5331988.7,0.131924
1,clean,102771,3900,5331988.7,0.131924


## 2. Monthly KPI Trends

In [4]:
monthly = monthly_kpis(df)
monthly.tail()

,purchase_month,orders,customers,sales,avg_order_value,promo_usage_rate,churn_rate
31,2024-08-01,4130,2297,204026.2,49.401017,0.139709,0.073574
32,2024-09-01,2299,1582,116707.3,50.764376,0.125272,0.055626
33,2024-10-01,2558,1771,123610.5,48.323104,0.134871,0.030491
34,2024-11-01,3644,2171,237547.2,65.188584,0.125960,0.003224
35,2024-12-01,4406,2296,284590.4,64.591557,0.125965,0.000000


In [5]:
save_monthly_trend(monthly, FIGURE_DIR / 'monthly_sales_promo_usage.png')

## 3. Promo Code Analysis

In [6]:
promo = promo_comparison(df)
promo

,promo_segment,orders,customers,sales,avg_order_value,previous_purchases_avg,churn_rate
0,No Promo,89213,3900,4632959.1,51.931435,33.094672,0.073678
1,Promo,13558,3476,699029.6,51.558460,34.424325,0.071987


In [7]:
save_promo_comparison(promo, FIGURE_DIR / 'promo_vs_no_promo.png')

## 4. First Observed Cohort Retention

In [8]:
retention = cohort_retention(df)
matrix = cohort_matrix(retention)
matrix.head()

cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35
first_observed_purchase_month,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2022-01,1.0,0.400463,0.487654,0.520062,0.518519,0.495370,0.479167,0.533179,0.399691,0.430556,0.542438,0.629630,0.492284,0.462963,0.533951,0.541667,0.544753,0.559414,0.556327,0.583333,0.461420,0.496914,0.575617,0.689815,0.573302,0.544753,0.614198,0.635802,0.653549,0.617284,0.648920,0.662809,0.497685,0.567901,0.675154,0.736111
2022-02,1.0,0.472335,0.466937,0.489879,0.499325,0.483131,0.515520,0.352227,0.377868,0.524966,0.596491,0.429150,0.431849,0.514170,0.522267,0.530364,0.543860,0.556005,0.576248,0.452092,0.446694,0.554656,0.622132,0.492578,0.473684,0.591093,0.620783,0.626181,0.596491,0.601889,0.649123,0.461538,0.511471,0.597841,0.649123,NaN
2022-03,1.0,0.506767,0.542857,0.478195,0.482707,0.506767,0.287218,0.312782,0.421053,0.461654,0.323308,0.315789,0.569925,0.580451,0.562406,0.514286,0.529323,0.541353,0.318797,0.374436,0.451128,0.511278,0.384962,0.409023,0.621053,0.639098,0.630075,0.606015,0.587970,0.654135,0.400000,0.418045,0.532331,0.530827,NaN,NaN
2022-04,1.0,0.403409,0.414773,0.400568,0.446023,0.272727,0.241477,0.357955,0.420455,0.272727,0.247159,0.440341,0.457386,0.502841,0.497159,0.471591,0.491477,0.258523,0.318182,0.414773,0.451705,0.377841,0.394886,0.528409,0.517045,0.571023,0.556818,0.531250,0.528409,0.386364,0.389205,0.497159,0.491477,NaN,NaN,NaN
2022-05,1.0,0.462617,0.429907,0.387850,0.294393,0.242991,0.383178,0.397196,0.275701,0.280374,0.429907,0.420561,0.392523,0.485981,0.420561,0.500000,0.266355,0.280374,0.345794,0.425234,0.415888,0.401869,0.485981,0.495327,0.495327,0.514019,0.500000,0.523364,0.308411,0.397196,0.495327,0.485981,NaN,NaN,NaN,NaN


In [9]:
save_cohort_heatmap(matrix, FIGURE_DIR / 'cohort_retention_heatmap.png')

## 5. RFM Segmentation

In [10]:
rfm = rfm_segments(df)
latest_inactivity = build_inactivity_snapshot(df, cutoff='2024-06-30', horizon_days=180)[['customer_id', 'future_inactive_180d']]
rfm = rfm.merge(latest_inactivity, on='customer_id', how='left')
segments = segment_summary(rfm)
segments

,segment,customers,avg_recency,avg_frequency,avg_monetary,avg_order_value,promo_usage_rate,churn_rate,future_inactive_180d_rate
1,Dormant / Churn Risk,952,129.155462,9.890756,503.763235,51.054992,0.074624,0.165966,0.218124
4,Regular,898,29.959911,17.842984,915.447550,52.254539,0.112035,0.093541,0.010067
3,Potential Loyalist,881,13.031782,32.559591,1686.556754,51.890458,0.134220,0.000000,0.001135
2,High Value,851,5.336075,42.206816,2229.695417,52.869224,0.138608,0.000000,0.000000
0,At Risk High Spender,318,55.078616,40.028302,2034.612893,50.984801,0.138144,0.289308,0.003145


In [11]:
save_segment_matrix(segments, FIGURE_DIR / 'segment_strategy_matrix.png')

## 6. Promo Sensitivity

促销敏感度作为独立维度，不再混入 RFM segment。

In [12]:
promo_summary = promo_sensitivity_summary(rfm)
promo_summary

,promo_sensitivity,customers,avg_recency,avg_frequency,avg_monetary,avg_order_value,promo_usage_rate,churn_rate,future_inactive_180d_rate
1,Moderate Promo,2352,35.531888,27.387755,1421.179719,51.836361,0.122575,0.079082,0.021259
0,High Promo,1124,32.209075,32.590747,1689.824377,51.908930,0.145451,0.078292,0.019573
2,No Promo,424,150.054245,4.063679,212.291038,52.322028,0.000000,0.141509,0.350120


## 7. Time-Split 180-Day Inactivity Risk Model

模型使用 cutoff 前历史行为生成特征，并用 cutoff 后 180 天是否无购买生成标签，避免全周期特征造成时间泄漏。模型结果只用于解释风险信号，不作为生产级预测系统。

In [13]:
importance, model_metrics = train_inactivity_risk_model(df)
importance.head(10)

,feature,importance
0,num__previous_purchases,0.252488
1,num__active_months,0.116031
2,num__frequency,0.111911
3,num__monetary,0.101257
4,num__promo_orders,0.067744
5,num__orders_per_active_month,0.050505
6,num__promo_usage_rate,0.050113
7,num__distinct_categories,0.041753
8,num__recency,0.037396
9,num__has_promo_order,0.034975


In [14]:
{
    'model_level': model_metrics['model_level'],
    'train_cutoff': model_metrics['train_cutoff'],
    'test_cutoff': model_metrics['test_cutoff'],
    'test_positive_rate': model_metrics['test_positive_rate'],
    'roc_auc': model_metrics['roc_auc'],
    'inactive_f1': model_metrics['classification_report']['1']['f1-score'],
}

{'model_level': 'customer_time_split', 'train_cutoff': '2023-06-30', 'test_cutoff': '2023-12-31', 'test_positive_rate': 0.052495474528057924, 'roc_auc': 0.9120748811495688, 'inactive_f1': 0.36771300448430494}